# Deteção de anomalias temporais nas séries de consumo energético

Este notebook identifica anomalias diretamente nas séries temporais de consumo energético, analisando a evolução ao longo do tempo.

Pretende-se:
- aplicar deteção a TODOS os CPEs (não apenas a uma amostra);
- comparar múltiplos métodos de deteção temporal;
- analisar sensibilidade a hiperparâmetros;
- medir tempos de execução (benchmarking);
- relacionar anomalias temporais com clusters e anomalias de features;
- analisar eventos específicos (ex: apagão 28/abril/2025).

In [ ]:
# Imports e configuração

import zipfile
import time
from pathlib import Path
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
 
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from statsmodels.tsa.seasonal import STL
 
plt.rcParams["figure.figsize"] = (12, 4)
sns.set_theme(style="whitegrid")
 
base_dir = Path("../results")
clustering_dir = base_dir / "clustering"
anomalies_dir = base_dir / "anomalies_temporal"
plots_dir = anomalies_dir / "plots"
 
anomalies_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Carregamento dos dados

repo_datasets_path = Path("../PM2025-TSAnomalyDetection/example-timeseries")
zip_path = repo_datasets_path / "consumo15m_11_2025.csv.zip"
 
with zipfile.ZipFile(zip_path) as z:
    with z.open(z.namelist()[0]) as f:
        df_energia = pd.read_csv(f)
 
df_energia["tstamp"] = pd.to_datetime(df_energia["tstamp"])
df_energia = df_energia.sort_values(["CPE", "tstamp"]).reset_index(drop=True)
 
print("Shape do dataset:", df_energia.shape)
print("Número de CPE:", df_energia["CPE"].nunique())
print("Período:", df_energia["tstamp"].min(), "→", df_energia["tstamp"].max())
 
# Carregar resultados dos notebooks anteriores
clusters = pd.read_csv(clustering_dir / "clusters_cpe.csv", index_col=0)
 
todos_cpes = df_energia["CPE"].unique().tolist()
print(f"\nTotal de CPEs a processar: {len(todos_cpes)}")
 
print(f"\nDistribuição por cluster:")
display(clusters["cluster"].value_counts())

In [ ]:
# Preparação das séries temporais

def preparar_serie(df, cpe):
    """
    Prepara uma série temporal individual:
    - Remove duplicados (agrega pela média)
    - Garante frequência regular de 15 min
    - Interpola gaps curtos (até 2h = 8 intervalos)
    - Deixa NaN em gaps longos (não interpolar cegamente)
    """
    df_cpe = df[df["CPE"] == cpe].copy()
    df_cpe = df_cpe.sort_values("tstamp")
    
    # Agregar duplicados
    df_cpe = (
        df_cpe.groupby("tstamp", as_index=False)["PotActiva"]
        .mean()
    )
    
    df_cpe = df_cpe.set_index("tstamp").sort_index()
    
    # Garantir frequência regular
    df_cpe = df_cpe.asfreq("15min")
    
    # Interpolar apenas gaps curtos (até 8 intervalos = 2h)
    df_cpe["PotActiva"] = df_cpe["PotActiva"].interpolate(limit=8)
    
    return df_cpe
 
# Preparar todas as séries
print("Preparando séries temporais...")
series_dict = {}
for i, cpe in enumerate(todos_cpes):
    series_dict[cpe] = preparar_serie(df_energia, cpe)
    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{len(todos_cpes)} séries preparadas")
 
print(f"  {len(series_dict)} séries preparadas ✓")
 
# Verificar qualidade
for cpe in list(series_dict.keys())[:3]:
    s = series_dict[cpe]
    pct_nan = s["PotActiva"].isna().mean() * 100
    print(f"  {cpe}: {len(s)} pontos, {pct_nan:.1f}% NaN após interpolação")

# Observações

A interpolação é limitada a gaps de até 2h (8 intervalos de 15min). Gaps mais longos são mantidos como NaN para evitar introduzir artefactos.

In [ ]:
# Métodos de deteção de anomalias temporais
 
# --- Rolling Z-Score ---
def detectar_zscore(df, window=96, threshold=3):
    """
    Rolling Z-Score: deteta pontos que desviam significativamente
    do comportamento recente.
    window=96 → 1 dia (dados de 15min)
    threshold=3 → |z| > 3 é anomalia
    """
    df = df.copy()
    df["rolling_mean"] = df["PotActiva"].rolling(window=window, min_periods=window//2).mean()
    df["rolling_std"] = df["PotActiva"].rolling(window=window, min_periods=window//2).std()
    
    df["z_score"] = np.where(
        df["rolling_std"] > 1e-6,
        (df["PotActiva"] - df["rolling_mean"]) / df["rolling_std"],
        0
    )
    
    df["anomalia"] = (df["z_score"].abs() > threshold).astype(int)
    return df
 
# --- STL Decomposition + Residuals ---
def detectar_stl(df, period=96, threshold=3):
    """
    STL Decomposition: separa tendência, sazonalidade e resíduo.
    Anomalias são pontos com resíduos extremos.
    period=96 → sazonalidade diária
    """
    df = df.copy()
    serie = df["PotActiva"].dropna()
    
    if len(serie) < period * 3:
        df["anomalia"] = 0
        df["residuo"] = np.nan
        return df
    
    try:
        stl = STL(serie, period=period, robust=True)
        result = stl.fit()
        
        residuo = result.resid
        residuo_std = residuo.std()
        residuo_mean = residuo.mean()
        
        z_residuo = np.where(
            residuo_std > 1e-6,
            (residuo - residuo_mean) / residuo_std,
            0
        )
        
        df["residuo"] = np.nan
        df.loc[residuo.index, "residuo"] = residuo
        df["tendencia"] = np.nan
        df.loc[result.trend.index, "tendencia"] = result.trend
        df["sazonalidade"] = np.nan
        df.loc[result.seasonal.index, "sazonalidade"] = result.seasonal
        
        df["anomalia"] = 0
        df.loc[residuo.index, "anomalia"] = (np.abs(z_residuo) > threshold).astype(int)
        
    except Exception as e:
        df["anomalia"] = 0
        df["residuo"] = np.nan
    
    return df
 
# --- Isolation Forest em janelas deslizantes ---
def detectar_iforest_temporal(df, window=96, step=48, contamination=0.05):
    """
    Isolation Forest aplicado a janelas temporais com features locais.
    window=96 → janela de 1 dia
    step=48 → avança meio dia
    """
    df = df.copy()
    df["anomalia"] = 0
    
    serie = df["PotActiva"].values
    n = len(serie)
    
    if n < window * 2:
        return df
    
    # Extrair features por janela
    janelas = []
    indices = []
    
    for start in range(0, n - window, step):
        end = start + window
        w = serie[start:end]
        
        if np.isnan(w).sum() > window * 0.3:
            continue
        
        w_clean = w[~np.isnan(w)]
        if len(w_clean) < window // 2:
            continue
        
        janelas.append({
            "mean": np.mean(w_clean),
            "std": np.std(w_clean),
            "min": np.min(w_clean),
            "max": np.max(w_clean),
            "range": np.max(w_clean) - np.min(w_clean),
            "slope": np.polyfit(np.arange(len(w_clean)), w_clean, 1)[0] if len(w_clean) > 1 else 0,
        })
        indices.append((start, end))
    
    if len(janelas) < 10:
        return df
    
    X_windows = pd.DataFrame(janelas)
    
    scaler = StandardScaler()
    X_windows_scaled = scaler.fit_transform(X_windows)
    
    iso = IsolationForest(contamination=contamination, random_state=42)
    labels = iso.fit_predict(X_windows_scaled)
    
    for (start, end), label in zip(indices, labels):
        if label == -1:
            df.iloc[start:end, df.columns.get_loc("anomalia")] = 1
    
    return df

In [ ]:
# Aplicação a todos os CPEs

print("APLICAÇÃO DOS MÉTODOS A TODOS OS CPEs")
 
metodos = {
    "Z-Score": detectar_zscore,
    "STL": detectar_stl,
    "IForest_Temporal": detectar_iforest_temporal,
}
 
resultados = {m: {} for m in metodos}
tempos = {m: [] for m in metodos}
 
for i, cpe in enumerate(todos_cpes):
    df_cpe = series_dict[cpe]
    
    for method_name, method_func in metodos.items():
        t0 = time.time()
        
        if method_name == "Z-Score":
            resultado = method_func(df_cpe, window=96, threshold=3)
        elif method_name == "STL":
            resultado = method_func(df_cpe, period=96, threshold=3)
        elif method_name == "IForest_Temporal":
            resultado = method_func(df_cpe, window=96, step=48, contamination=0.05)
        
        elapsed = time.time() - t0
        tempos[method_name].append(elapsed)
        resultados[method_name][cpe] = resultado
    
    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{len(todos_cpes)} CPEs processados")
 
print(f"  {len(todos_cpes)} CPEs processados ✓")

In [ ]:
# Benchmarking de tempos de execução

print("BENCHMARKING - TEMPOS DE EXECUÇÃO")
 
df_tempos = pd.DataFrame({
    method: {
        "Total (s)": sum(t),
        "Média por CPE (ms)": np.mean(t) * 1000,
        "Mediana por CPE (ms)": np.median(t) * 1000,
        "Max por CPE (ms)": np.max(t) * 1000,
    }
    for method, t in tempos.items()
}).T
 
display(df_tempos.round(2))
df_tempos.to_csv(anomalies_dir / "benchmarking_tempos.csv")
 
plt.figure(figsize=(8, 4))
plt.bar(df_tempos.index, df_tempos["Total (s)"], color=["#0D7C66", "#14A38B", "#2980B9"])
plt.title("Tempo total de execução por método")
plt.ylabel("Tempo (segundos)")
plt.tight_layout()
plt.savefig(plots_dir / "benchmarking_tempos.png", dpi=150)
plt.show()

In [ ]:
# Resumo de anomalias por CPE e método

print("RESUMO DE ANOMALIAS POR CPE")
 
resumo_rows = []
 
for cpe in todos_cpes:
    row = {"CPE": cpe}
    
    for method_name in metodos:
        df_res = resultados[method_name][cpe]
        total = len(df_res)
        n_anom = df_res["anomalia"].sum()
        row[f"{method_name}_n"] = n_anom
        row[f"{method_name}_pct"] = n_anom / total * 100 if total > 0 else 0
    
    resumo_rows.append(row)
 
df_resumo = pd.DataFrame(resumo_rows).set_index("CPE")
df_resumo = df_resumo.join(clusters[["cluster"]])
 
print("\nTop 10 CPEs com mais anomalias (Z-Score):")
display(df_resumo.sort_values("Z-Score_pct", ascending=False).head(10))

In [ ]:
# Concordância entre métodos temporais

print("CONCORDÂNCIA ENTRE MÉTODOS TEMPORAIS")
 
THRESHOLD_PCT = 2.0  # CPE com >2% anomalias é considerado "instável"
 
for method in metodos:
    df_resumo[f"{method}_instavel"] = (df_resumo[f"{method}_pct"] > THRESHOLD_PCT).astype(int)
 
instavel_cols = [f"{m}_instavel" for m in metodos]
df_resumo["n_metodos_instavel"] = df_resumo[instavel_cols].sum(axis=1)
 
print(f"\nCPEs classificados como instáveis (>{THRESHOLD_PCT}% anomalias) por método:")
for method in metodos:
    n = df_resumo[f"{method}_instavel"].sum()
    print(f"  {method}: {n} CPEs instáveis")
 
print(f"\nDistribuição de concordância:")
display(df_resumo["n_metodos_instavel"].value_counts().sort_index())
 
cpes_instavel_consenso = df_resumo[df_resumo["n_metodos_instavel"] >= 2].index.tolist()
print(f"\nCPEs instáveis por consenso (≥2 métodos): {len(cpes_instavel_consenso)}")

In [ ]:
# Análise por cluster

print("ANOMALIAS TEMPORAIS POR CLUSTER")
 
for method in metodos:
    pct_col = f"{method}_pct"
    print(f"\n{method} — taxa média de anomalias por cluster:")
    display(
        df_resumo.groupby("cluster")[pct_col]
        .agg(["mean", "median", "std", "count"])
        .round(3)
    )
 
fig, axes = plt.subplots(1, len(metodos), figsize=(5*len(metodos), 4))
if len(metodos) == 1:
    axes = [axes]
 
for ax, method in zip(axes, metodos):
    pct_col = f"{method}_pct"
    sns.boxplot(data=df_resumo, x="cluster", y=pct_col, ax=ax, palette="tab10")
    ax.set_title(method)
    ax.set_ylabel("% anomalias")
 
plt.suptitle("Distribuição de anomalias temporais por cluster", fontweight="bold")
plt.tight_layout()
plt.savefig(plots_dir / "anomalias_temporais_por_cluster.png", dpi=150)
plt.show()

In [ ]:
# Visualização de exemplos representativos

print("EXEMPLOS REPRESENTATIVOS")
 
# CPE mais estável e mais instável
cpe_estavel = df_resumo["Z-Score_pct"].idxmin()
cpe_instavel = df_resumo["Z-Score_pct"].idxmax()
 
# 1 por cluster (mediana)
cpes_por_cluster = {}
for cluster_id in sorted(df_resumo[df_resumo["cluster"] != "outlier"]["cluster"].unique()):
    cpes_cluster = df_resumo[df_resumo["cluster"] == cluster_id]
    mediana = cpes_cluster["Z-Score_pct"].median()
    cpe_repr = (cpes_cluster["Z-Score_pct"] - mediana).abs().idxmin()
    cpes_por_cluster[cluster_id] = cpe_repr
 
cpes_visualizar = list(set([cpe_estavel, cpe_instavel] + list(cpes_por_cluster.values())))
 
print(f"CPE mais estável: {cpe_estavel} ({df_resumo.loc[cpe_estavel, 'Z-Score_pct']:.2f}%)")
print(f"CPE mais instável: {cpe_instavel} ({df_resumo.loc[cpe_instavel, 'Z-Score_pct']:.2f}%)")
for cl, cpe in cpes_por_cluster.items():
    print(f"CPE representativo cluster {cl}: {cpe}")
 
for cpe in cpes_visualizar:
    cluster = df_resumo.loc[cpe, "cluster"] if cpe in df_resumo.index else "?"
    
    fig, axes = plt.subplots(len(metodos), 1, figsize=(14, 3*len(metodos)), sharex=True)
    
    for ax, method in zip(axes, metodos):
        df_res = resultados[method][cpe]
        
        ax.plot(df_res.index, df_res["PotActiva"], linewidth=0.5, alpha=0.8, label="Consumo")
        
        anomalias = df_res[df_res["anomalia"] == 1]
        if len(anomalias) > 0:
            ax.scatter(anomalias.index, anomalias["PotActiva"],
                       color="red", s=8, zorder=5, label=f"Anomalias ({len(anomalias)})")
        
        ax.set_ylabel("PotActiva")
        ax.set_title(f"{method}: {len(anomalias)} anomalias ({len(anomalias)/len(df_res)*100:.1f}%)", fontsize=10)
        ax.legend(loc="upper right", fontsize=8)
        ax.grid(True, alpha=0.3)
    
    axes[-1].set_xlabel("Tempo")
    plt.suptitle(f"{cpe} (Cluster {cluster})", fontweight="bold", fontsize=12)
    plt.tight_layout()
    plt.savefig(plots_dir / f"anomalias_temporais_{cpe}.png", dpi=150)
    plt.show()

In [ ]:
# Análise de eventos específicos

print("ANÁLISE DE EVENTOS ESPECÍFICOS")
 
data_apagao = pd.Timestamp("2025-04-28")
janela_inicio = data_apagao - pd.Timedelta(days=2)
janela_fim = data_apagao + pd.Timedelta(days=2)
 
print(f"\nEvento: Apagão de 28/abril/2025")
print(f"Janela de análise: {janela_inicio.date()} → {janela_fim.date()}")
 
anom_apagao = {}
for cpe in todos_cpes:
    for method in metodos:
        df_res = resultados[method][cpe]
        mask = (df_res.index >= janela_inicio) & (df_res.index <= janela_fim)
        df_periodo = df_res.loc[mask]
        
        if len(df_periodo) > 0:
            n_anom = df_periodo["anomalia"].sum()
            if cpe not in anom_apagao:
                anom_apagao[cpe] = {}
            anom_apagao[cpe][method] = n_anom
 
df_apagao = pd.DataFrame(anom_apagao).T
df_apagao["total"] = df_apagao.sum(axis=1)
df_apagao = df_apagao.sort_values("total", ascending=False)
 
n_afetados = (df_apagao["total"] > 0).sum()
print(f"\nCPEs com anomalias detetadas no período do apagão: {n_afetados}/{len(todos_cpes)}")
print("\nTop 10 CPEs mais afetados:")
display(df_apagao.head(10))
 
# Visualizar um CPE durante o apagão
cpe_apagao = df_apagao.index[0] if len(df_apagao) > 0 else todos_cpes[0]
 
fig, axes = plt.subplots(len(metodos), 1, figsize=(12, 3*len(metodos)), sharex=True)
 
for ax, method in zip(axes, metodos):
    df_res = resultados[method][cpe_apagao]
    mask = (df_res.index >= janela_inicio) & (df_res.index <= janela_fim)
    df_periodo = df_res.loc[mask]
    
    ax.plot(df_periodo.index, df_periodo["PotActiva"], linewidth=1, label="Consumo")
    
    anomalias = df_periodo[df_periodo["anomalia"] == 1]
    if len(anomalias) > 0:
        ax.scatter(anomalias.index, anomalias["PotActiva"], color="red", s=15, zorder=5, label="Anomalias")
    
    ax.axvline(data_apagao, color="black", linestyle="--", alpha=0.5, label="28/abril")
    ax.set_ylabel("PotActiva")
    ax.set_title(f"{method}", fontsize=10)
    ax.legend(loc="upper right", fontsize=8)
    ax.grid(True, alpha=0.3)
 
axes[-1].set_xlabel("Tempo")
plt.suptitle(f"Apagão 28/abril/2025 — {cpe_apagao}", fontweight="bold")
plt.tight_layout()
plt.savefig(plots_dir / "evento_apagao.png", dpi=150)
plt.show()

In [ ]:
# Sensibilidade a hiperparâmetros (Z-Score)

print("SENSIBILIDADE A HIPERPARÂMETROS (Z-Score)")
 
cpe_teste = list(cpes_por_cluster.values())[0]
df_teste = series_dict[cpe_teste]
 
windows = [48, 96, 192, 672]  # 12h, 1d, 2d, 1 semana
thresholds = [2.0, 2.5, 3.0, 3.5, 4.0]
 
sensibilidade = []
for w in windows:
    for t in thresholds:
        resultado = detectar_zscore(df_teste, window=w, threshold=t)
        n_anom = resultado["anomalia"].sum()
        pct = n_anom / len(resultado) * 100
        sensibilidade.append({
            "window": w,
            "window_label": f"{w*15/60:.0f}h",
            "threshold": t,
            "n_anomalias": n_anom,
            "pct_anomalias": pct
        })
 
df_sens = pd.DataFrame(sensibilidade)
 
pivot_sens = df_sens.pivot(index="threshold", columns="window_label", values="pct_anomalias")
 
plt.figure(figsize=(8, 5))
sns.heatmap(pivot_sens, annot=True, fmt=".2f", cmap="YlOrRd")
plt.title(f"Sensibilidade do Z-Score (CPE: {cpe_teste})\n% anomalias por window × threshold")
plt.ylabel("Threshold (|z|)")
plt.xlabel("Janela")
plt.tight_layout()
plt.savefig(plots_dir / "sensibilidade_zscore.png", dpi=150)
plt.show()

In [ ]:
# STL Decomposition — visualização detalhada

print("DECOMPOSIÇÃO STL — EXEMPLO DETALHADO")
 
cpe_stl = list(cpes_por_cluster.values())[0]
serie_stl = series_dict[cpe_stl]["PotActiva"].dropna()
 
if len(serie_stl) > 96 * 3:
    stl_result = STL(serie_stl, period=96, robust=True).fit()
    
    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    
    axes[0].plot(serie_stl.index, serie_stl.values, linewidth=0.5)
    axes[0].set_ylabel("Original")
    axes[0].set_title(f"Decomposição STL — {cpe_stl}")
    
    axes[1].plot(stl_result.trend.index, stl_result.trend.values, linewidth=1, color="#0D7C66")
    axes[1].set_ylabel("Tendência")
    
    axes[2].plot(stl_result.seasonal.index, stl_result.seasonal.values, linewidth=0.5, color="#2980B9")
    axes[2].set_ylabel("Sazonalidade")
    
    axes[3].plot(stl_result.resid.index, stl_result.resid.values, linewidth=0.5, color="#636E72")
    resid_std = stl_result.resid.std()
    resid_mean = stl_result.resid.mean()
    axes[3].axhline(resid_mean + 3*resid_std, color="red", linestyle="--", alpha=0.5)
    axes[3].axhline(resid_mean - 3*resid_std, color="red", linestyle="--", alpha=0.5)
    axes[3].set_ylabel("Resíduo")
    axes[3].set_xlabel("Tempo")
    
    for ax in axes:
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(plots_dir / "stl_decomposicao.png", dpi=150)
    plt.show()

In [ ]:
# Tabela comparativa final

print("TABELA COMPARATIVA FINAL")
 
comparacao_final = pd.DataFrame({
    "Método": list(metodos.keys()),
    "Tempo total (s)": [sum(tempos[m]) for m in metodos],
    "Média anomalias/CPE (%)": [df_resumo[f"{m}_pct"].mean() for m in metodos],
    "Mediana anomalias/CPE (%)": [df_resumo[f"{m}_pct"].median() for m in metodos],
    "CPEs instáveis (>2%)": [df_resumo[f"{m}_instavel"].sum() for m in metodos],
}).set_index("Método")
 
display(comparacao_final.round(3))
comparacao_final.to_csv(anomalies_dir / "comparacao_metodos_temporais.csv")

In [ ]:
# 13. Exportação de resultados

df_resumo.to_csv(anomalies_dir / "anomalias_temporais_resumo.csv")
print("\nResumo temporal guardado:", anomalies_dir / "anomalias_temporais_resumo.csv")
 
df_tempos.to_csv(anomalies_dir / "benchmarking_tempos.csv")
print("Benchmarking guardado:", anomalies_dir / "benchmarking_tempos.csv")
 
print("\nGráficos guardados em:", plots_dir)

# Conclusões

A deteção de anomalias temporais foi aplicada a TODOS os CPEs com 3 métodos: Rolling Z-Score, STL Decomposition e IsolationForest em janelas deslizantes.

Principais conclusões:
- Os métodos têm diferentes perfis de sensibilidade e tempo de execução
- A concordância entre métodos identifica anomalias mais robustas
- A taxa de anomalias varia entre clusters, validando a segmentação
- A análise de sensibilidade justifica a escolha dos hiperparâmetros
- Eventos conhecidos (apagão) são detetados consistentemente

Para trabalho futuro:
- Explorar autoencoders (LSTM / Conv1D)
- Integrar variáveis externas (meteorologia)
- Desenvolver sistema de alertas em tempo real